# Numerical solver for the discrete space and structured model

### Imports and setup

In [ ]:
import numpy as np
import warnings
from scipy.integrate import solve_ivp
from scipy.sparse import kron, diags
from dataclasses import dataclass
import matplotlib.pyplot as plt

@dataclass
class Params:
    M: int
    N: int
    kplus: float
    kminus: float
    beta: float
    D_min: float
    gamma: float
    sigma_b: float
    sigma_max: float
    V_tot: float    
    delta_v: float  
    theta: float
    debug: bool = False
    tol: float = 1e-6

### Model class definition

In [ ]:
class MacrophageModel:
    def __init__(self, p: Params):
        self.p = p
        # Cache static arrays here to save time during ODE integration
        self.v = 1.0 + np.arange(p.N + 1) * p.delta_v
        self.n_array = np.arange(p.N + 1)
        self.n_decay = 1.0 - self.n_array / p.N
        self.n_growth = self.n_array / p.N
        
        # Pre-compute the Jacobian sparsity matrix 
        self.jac_sparsity = self._build_sparsity_matrix()

    def compute_phi(self, u: np.ndarray) -> np.ndarray:
        """
        No-voids constraint: phi_i = 1 - sum_n v_n u[i,n]
        """
        phi = 1.0 - np.sum(u * self.v, axis=-1)
        
        min_phi = np.min(phi)
        if min_phi < -1e-6:
            warnings.warn(f"Severe physical violation: phi dropped to {min_phi:.2e}. "
                          f"Grid is over-saturated or solver step is too large.")
        
        return np.clip(phi, 0.0, 1.0)

    def compute_VL(self, u: np.ndarray):
        if u.ndim == 3:
            return (self.p.theta / self.p.M) * np.sum(u * self.n_array, axis=(1, 2))
        else:
            return (self.p.theta / self.p.M) * np.sum(u * self.n_array)

    def compute_mean_lipid_load_spatial(self, U: np.ndarray) -> np.ndarray:
        """
        Computes:  sum_n(n * u_{i,n}) / sum_n(u_{i,n})
        U shape: (Time, M+1, N+1)
        """
        # Numerator: Sum of packets (n * u_{i,n})
        # We reuse self.n_array instead of recreating it!
        total_fraction = np.sum(U * self.n_array, axis=2) 
        
        # Denominator: Total macrophage concentration (sum of u_{i,n})
        total_concentration = np.sum(U, axis=2)
        
        # Avoid division by zero where concentration is extremely small or zero
        safe_denominator = np.where(total_concentration < 1e-12, 1.0, total_concentration)
        
        # Calculate the mean load
        mean_load = (total_fraction / safe_denominator)
        
        # Where concentration is zero, set load to 0
        return np.where(total_concentration < 1e-12, 0.0, mean_load)

    def compute_mean_lipid_load_domain(self, U: np.ndarray) -> np.ndarray:
        """
        Computes the domain-wide mean lipid volume fraction over time.
        Formula: sum_{i,n} ( n * u_{i,n} ) / sum_{i,n} (u_{i,n})
        U shape: (Time, M+1, N+1)
        Returns: 1D array of shape (Time,)
        """
        # Numerator: Weight the fraction by the cell density and sum across SPACE and CLASSES
        total_domain_fraction = np.sum(U * self.n_array, axis=(1, 2)) 
        
        # Denominator: Total macrophage concentration across the entire domain
        total_domain_macrophages = np.sum(U, axis=(1, 2))
        
        # Avoid division by zero
        safe_denominator = np.where(total_domain_macrophages < 1e-12, 1.0, total_domain_macrophages)
        
        # Calculate the domain-wide mean fraction
        mean_domain_fraction = total_domain_fraction / safe_denominator
        
        # Where concentration is zero, set fraction to 0.0
        return np.where(total_domain_macrophages < 1e-12, 0.0, mean_domain_fraction)

    def _build_sparsity_matrix(self):
        """
        Generates the Jacobian sparsity matrix for the atherosclerosis model.
        """
        p = self.p
        spatial_diags = [np.ones(p.M), np.ones(p.M + 1), np.ones(p.M)]
        spatial_sparse = diags(spatial_diags, offsets=[-1, 0, 1], format='csr')
        lipid_dense = np.ones((p.N + 1, p.N + 1))
        jac_sparsity = kron(spatial_sparse, lipid_dense, format='lil')
        jac_sparsity[0, :] = 1
        return jac_sparsity.tocsr()

    # Function that computes the right-hand side of the ODE system
    def rhs(self, t: float, y: np.ndarray) -> np.ndarray:
        M, N, p = self.p.M, self.p.N, self.p
        u = y.reshape((M + 1, N + 1))

        phi = self.compute_phi(u)
        VLt = self.compute_VL(u)

        # THE HEAVISIDE MATRIX IMPLEMENTATION
        capacity_val = (p.V_tot / M) * phi[:, None] - self.v[None, :]
        H = (capacity_val >= 0).astype(float)
        phi_H = phi[:, None] * H

        du = np.zeros_like(u)

        # 1) Vectorized interior in i for n = 0 (i = 1,...,M-1)
        if M >= 2:
            du[1:M, 0] = (
                - N * p.kplus * phi[1:M] * u[1:M, 0]
                + p.kminus * u[1:M, 1] 
                + M**2 * (phi_H[1:M, 0] * (u[0:M-1, 0] + u[2:M+1, 0])
                    - u[1:M, 0] * (phi_H[0:M-1, 0] + phi_H[2:M+1, 0])
                )
                - p.beta * u[1:M, 0]
            )

        # 2) Vectorized interior in i for n = N (i = 1,...,M-1)
        if M >= 2:
            du[1:M, N] = (
                p.kplus * phi[1:M] * u[1:M, N-1]
                - N * p.kminus * u[1:M, N]
                + M**2 * p.D_min * (
                    phi_H[1:M, N] * (u[0:M-1, N] + u[2:M+1, N])
                    - u[1:M, N] * (phi_H[0:M-1, N] + phi_H[2:M+1, N])
                )
                - p.beta * u[1:M, N]
            )

        # 3) Vectorized boundary row i = 0, for n = 1,...,N-1
        if N >= 2:
            n = np.arange(1, N)
            D_n = p.D_min + (1.0 - p.D_min) * self.n_decay[n]

            du[0, 1:N] = (
                N * p.kplus * phi[0] * (self.n_decay[n - 1] * u[0, 0:N-1] - self.n_decay[n] * u[0, 1:N])
                + N * p.kminus * (self.n_growth[n + 1] * u[0, 2:N+1] - self.n_growth[n] * u[0, 1:N])
                + M**2 * D_n * (phi_H[0, 1:N] * u[1, 1:N] - phi_H[1, 1:N] * u[0, 1:N])
                - p.beta * u[0, 1:N]
            )

            # 4) Vectorized boundary row i = M, for n = 1,...,N-1
            du[M, 1:N] = (
                N * p.kplus * phi[M] * (self.n_decay[n - 1] * u[M, 0:N-1] - self.n_decay[n] * u[M, 1:N])
                + N * p.kminus * (self.n_growth[n + 1] * u[M, 2:N+1] - self.n_growth[n] * u[M, 1:N])
                + M**2 * D_n * (phi_H[M, 1:N] * u[M-1, 1:N] - phi_H[M-1, 1:N] * u[M, 1:N] )
                - M * p.gamma * D_n * u[M, 1:N]
                - p.beta * u[M, 1:N]
            )

            # 5) Full interior block: 1 <= i <= M-1, 1 <= n <= N-1
            if M >= 2:
                D_n_2d = p.D_min + (1.0 - p.D_min) * self.n_decay[n][None, :]
                du[1:M, 1:N] = (
                    N * p.kplus * phi[1:M, None] * (
                        self.n_decay[n - 1][None, :] * u[1:M, 0:N-1]
                        - self.n_decay[n][None, :] * u[1:M, 1:N]
                    )
                    + N * p.kminus * (
                        self.n_growth[n + 1][None, :] * u[1:M, 2:N+1]
                        - self.n_growth[n][None, :] * u[1:M, 1:N]
                    )
                    + M**2 * D_n_2d * (
                        phi_H[1:M, 1:N] * (u[0:M-1, 1:N] + u[2:M+1, 1:N])
                        - u[1:M, 1:N] * (phi_H[0:M-1, 1:N] + phi_H[2:M+1, 1:N])
                    )
                    - p.beta * u[1:M, 1:N]
                )

        # 6) Corners
        du[0, 0] = (
            - N * p.kplus * phi[0] * u[0, 0]
            + p.kminus * u[0, 1]
            + M * (p.sigma_b + p.sigma_max * VLt / (1.0 + VLt)) * phi_H[0, 0]
            + M**2 * (phi_H[0, 0] * u[1, 0] - phi_H[1, 0] * u[0, 0])
            - p.beta * u[0, 0]
        )

        du[0, N] = (
            p.kplus * phi[0] * u[0, N-1]
            - N * p.kminus * u[0, N]
            + M**2 * p.D_min * (phi_H[0, N] * u[1, N] - phi_H[1, N] * u[0, N])
            - p.beta * u[0, N]
        )

        du[M, 0] = (
            - N * p.kplus * phi[M] * u[M, 0]
            + p.kminus * u[M, 1]
            + M**2 * (phi_H[M, 0] * u[M-1, 0] - phi_H[M-1, 0] * u[M, 0])
            - M * p.gamma * u[M, 0]
            - p.beta * u[M, 0]
        )

        du[M, N] = (
            p.kplus * phi[M] * u[M, N-1]
            - N * p.kminus * u[M, N]
            + M**2 * p.D_min * (phi_H[M, N] * u[M-1, N] - phi_H[M-1, N] * u[M, N])
            - M * p.gamma * p.D_min * u[M, N]
            - p.beta * u[M, N]
        )

        # if not np.isfinite(du).all():
        #   raise ValueError(f"Math exploded (NaN/Inf detected) at exactly t = {t}")

        return du.ravel()

    # Function to run the BDF solver and return results
    def solve(self, t_span: tuple, t_eval: np.ndarray, y0: np.ndarray = None):
        """
        Runs the BDF solver and returns the resulting 'U' 3D array (time, M+1, N+1) 
        and the full solver output object.
        """
        if y0 is None:
            # Default zero initial condition
            u0 = np.zeros((self.p.M + 1, self.p.N + 1))
            y0 = u0.ravel()

        sol = solve_ivp(
            fun=self.rhs,
            t_span=t_span,
            y0=y0,
            t_eval=t_eval,
            method="BDF",
            jac_sparsity=self.jac_sparsity,
            rtol=1e-6,
            atol=1e-9,
        )
        
        if not sol.success:
            warnings.warn(f"Solver failed: {sol.message}")
            
        U = sol.y.T.reshape((-1, self.p.M + 1, self.p.N + 1))
        return U, sol

# Function to compute useful quantities
def compute_diagnostics(self, U: np.ndarray) -> dict:
        """
        Takes the raw 3D U matrix from the solver and calculates all 
        spatial, structural, and domain-wide profiles.
        Returns a dictionary containing all diagnostic arrays.
        """
        p = self.p
        v = self.v
        
        # Helper conversions
        vol_to_number = p.V_tot / p.M

        # --- 1. SPATIAL PROFILES (shape: time, M+1) ---
        spatial = {
            "cell_density": np.sum(U, axis=2),
            "volume_fraction": np.sum(U * v, axis=2),
            "mean_lipid_load": self.compute_mean_lipid_load_spatial(U),
            "phi": self.compute_phi(U)
        }
        spatial["macrophage_number"] = vol_to_number * spatial["cell_density"]

        # --- 2. STRUCTURE PROFILES (shape: time, N+1) ---
        structure = {
            "cell_density": np.sum(U, axis=1),
            "volume_fraction": np.sum(U * v, axis=1)
        }
        structure["macrophage_number"] = vol_to_number * structure["cell_density"]

        # --- 3. TOTAL QUANTITIES OVER TIME (shape: time,) ---
        totals = {
            "cell_density": np.sum(U, axis=(1, 2)),
            "volume_fraction": np.sum(U * v, axis=(1, 2)),
            "mean_lipid_load": self.compute_mean_lipid_load_domain(U),
            "internalised_lipid_volume": self.compute_VL(U)
        }
        totals["macrophage_number"] = vol_to_number * totals["cell_density"]

        # Return everything bundled neatly
        return {
            "spatial": spatial,
            "structure": structure,
            "totals": totals
        }

def validate_results(self, U: np.ndarray, tolerance: float = 1e-5, strict: bool = False):
        """
        Runs physical sanity checks on the solver output.
        If strict=True, raises a ValueError on physical violations.
        Otherwise, raises a warning.
        """
        print("--- Running Post-Simulation Sanity Checks ---")
        
        # Check for NaNs or Infs (This should ALWAYS be a hard error)
        if not np.isfinite(U).all():
            raise ValueError("NUMERICAL EXPLOSION: NaNs or Infs detected in the output matrix!")
            
        # 1. Check for Non-negativity
        min_u = np.min(U)
        if min_u < -tolerance:
            msg = f"NON-NEGATIVITY VIOLATION: Minimum concentration is {min_u:.2e}."
            if strict: raise ValueError(msg)
            else: warnings.warn(msg)

        # 2. Check Volume Fraction (Phi) Bounds
        phi = self.compute_phi(U)
        min_phi, max_phi = np.min(phi), np.max(phi)
        
        if min_phi < -tolerance:
            msg = f"CAPACITY VIOLATION: Minimum phi dropped to {min_phi:.2e}."
            if strict: raise ValueError(msg)
            else: warnings.warn(msg)
            
        if max_phi > 1.0 + tolerance:
            msg = f"CAPACITY VIOLATION: Maximum phi exceeded 1.0 ({max_phi:.2e})."
            if strict: raise ValueError(msg)
            else: warnings.warn(msg)
            
        print("[\u2713] All physical constraints respected!")
        print("---------------------------------------------")

### Parameters and execution

In [ ]:
# Define your parameters
p = Params(
    M=50, N=100, kplus=1, kminus=1, beta=2, D_min=0.1, gamma=0.2,
    sigma_b=0.004, sigma_max=0.4, V_tot=100000, delta_v=0.005, theta=1, debug=True
)

t_final = 20.0

In [ ]:
# Instantiate the model
model = MacrophageModel(p)

# Setup Time
t_span = (0.0, t_final)
t_eval = np.linspace(t_span[0], t_span[1], 500)

# Run the solver!
print("Starting solver...")
U, sol = model.solve(t_span, t_eval)
print(f"Solver finished successfully: {sol.success}")

# Extract all data
results = model.compute_diagnostics(U)

### Plotting and analysis